In [ ]:
import sys
! pip install accelerate
!{sys.executable} -m pip install huggingface_hub
!{sys.executable} -m pip install transformers datasets

In [ ]:
from huggingface_hub import login

try:
    hf_token = ''
    login(token=hf_token)
    print("Successfully logged in to Hugging Face.")
except Exception as e:
    print(f"Failed to log in to Hugging Face. Please ensure 'HF_TOKEN' is set in Colab secrets and has the correct value. Error: {e}")

In [3]:
import torch
import torch.nn as nn

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.1-8B-Instruct", device_map='cuda')

In [ ]:
model.eval()

In [ ]:
# See the full tree
# print(model)

# For Llama-2, the FFN layers are at:
# model.model.layers[i].mlp.down_proj
# Let's verify:
for name, module in model.named_modules():
    if "down_proj" in name:
        print(name, "→", module)
        break
# Output: model.model.layers.0.mlp.down_proj → Linear(in=11008, out=4096)
#                                                        ^^^^ these are your neurons

In [4]:
class ActivationExtractor:
    def __init__(self, model):
        self.model = model
        self.activations = {}   # layer_idx -> tensor
        self._hooks = []

    def register_hooks(self):
        """Attach a hook to every down_proj layer."""
        # Remove old hook handles (does NOT clear activations)
        self.remove_hooks()
        # Clear activation buffer for this fresh run — done HERE, not in remove_hooks
        self.activations = {}

        for layer_idx, layer in enumerate(self.model.model.layers):
            def make_hook(idx):
                def hook_fn(module, input, output):
                    # input[0] shape: [batch, seq_len, intermediate_size]
                    self.activations[idx] = input[0].detach().cpu()
                return hook_fn

            handle = layer.mlp.down_proj.register_forward_hook(make_hook(layer_idx))
            self._hooks.append(handle)

    def remove_hooks(self):
        """Remove registered hook handles only.
        Activations are kept intentionally so callers can read them after
        unhooking. They are cleared at the START of register_hooks() instead.
        """
        for handle in self._hooks:
            handle.remove()
        self._hooks = []

In [5]:
def get_neuron_activations(extractor, tokenizer, prompt, response,
                            token_indices=None):
    # Tokenize prompt and response separately to know where response starts
    prompt_ids   = tokenizer.encode(prompt,   add_special_tokens=True)
    response_ids = tokenizer.encode(response, add_special_tokens=False)

    # Edge case: empty response (model generated nothing)
    if len(response_ids) == 0:
        return {}

    full_ids   = prompt_ids + response_ids
    input_tensor = torch.tensor([full_ids]).to(extractor.model.device)

    # Register fresh hooks (remove_hooks is called inside register_hooks)
    extractor.register_hooks()

    with torch.no_grad():
        extractor.model(input_tensor)   # ← use extractor.model, not global model

    extractor.remove_hooks()            # ← always remove after, not commented out

    # Slice to response token positions only
    response_start = len(prompt_ids)

    if token_indices is None:
        token_indices = list(range(response_start, len(full_ids)))

    # Another edge case: if token_indices is empty
    if len(token_indices) == 0:
        return {}

    sliced = {}
    for layer_idx, act in extractor.activations.items():
        # act: [1, seq_len, intermediate_size]
        sliced[layer_idx] = act[0, token_indices, :]

    return sliced

In [8]:
def compute_cett(layer_activations, method="mean"):
    """
    Reduce token-level activations to a single score per neuron per layer.

    Args:
        layer_activations: dict {layer_idx: tensor [num_tokens, intermediate_size]}
        method: "mean" averages absolute activations across tokens
    Returns:
        tensor of shape [num_layers * intermediate_size]  ← your feature vector
    """
    vectors = []
    for layer_idx in sorted(layer_activations.keys()):
        act = layer_activations[layer_idx]  # [num_tokens, intermediate_size]

        if method == "mean":
            # Mean of absolute values across token dimension
            score = act.abs().mean(dim=0)   # [intermediate_size]
        elif method == "max":
            score = act.abs().max(dim=0).values

        vectors.append(score)

    # Concatenate all layers → one flat feature vector
    return torch.cat(vectors, dim=0)   # [num_layers * intermediate_size]

In [9]:
import json
from pathlib import Path

def extract_dataset_activations(jsonl_path, output_dir, model, tokenizer,
                                  extractor):
    """
    Processes a .jsonl file where each line has:
      {"qid": "...", "prompt": "...", "response": "...",
       "label": 0 or 1,               ← 1 = hallucination
       "answer_token_indices": [...]}  ← from the repo's Step 2
    """
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    features, labels, qids = [], [], []

    with open(jsonl_path) as f:
        for line in f:
            sample = json.loads(line)

            token_indices = sample.get("answer_token_indices") or None

            layer_acts = get_neuron_activations(
                extractor,
                tokenizer,
                sample["prompt"],
                sample["response"],
                token_indices=token_indices
            )

            feature_vec = compute_cett(layer_acts, method="mean")
            features.append(feature_vec)
            labels.append(sample["label"])
            qids.append(sample["qid"])

    features = torch.stack(features)   # [N, num_layers * intermediate_size]
    labels = torch.tensor(labels)

    torch.save({"features": features, "labels": labels, "qids": qids},
               f"{output_dir}/activations.pt")
    print(f"Saved {len(features)} samples, feature dim = {features.shape[1]}")
    return features, labels

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
import numpy as np

def train_classifier(features, labels, penalty="l1", C=1.0):
    """
    features: torch.Tensor [N, D]
    labels:   torch.Tensor [N]
    penalty:  "l1" → sparse H-Neurons / "l2" → higher accuracy
    C:        inverse regularization strength. smaller = more sparsity
    """
    X = features.numpy().astype(np.float32)
    y = labels.numpy()

    # Standardize — critical for L1 to work properly
    # (neurons with large absolute values would otherwise dominate)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    clf = LogisticRegression(
        penalty=penalty,
        C=C,
        solver="liblinear",   # required for L1
        max_iter=1000,
        class_weight="balanced"
    )
    clf.fit(X_scaled, y)

    # Identify the H-Neurons
    coef = clf.coef_[0]           # shape [D]
    nonzero_mask = coef != 0
    h_neuron_count = nonzero_mask.sum()
    print(f"H-Neurons found: {h_neuron_count} / {len(coef)} "
          f"({100*h_neuron_count/len(coef):.3f}%)")

    return clf, scaler

def decode_h_neuron_indices(coef, num_layers, intermediate_size):
    """Convert flat weight indices back to (layer, neuron) pairs."""
    nonzero_flat = np.where(coef != 0)[0]
    h_neurons = []
    for flat_idx in nonzero_flat:
        layer = flat_idx // intermediate_size
        neuron = flat_idx % intermediate_size
        h_neurons.append({
            "flat_idx": int(flat_idx),
            "layer": int(layer),
            "neuron": int(neuron),
            "weight": float(coef[flat_idx])
        })
    return sorted(h_neurons, key=lambda x: abs(x["weight"]), reverse=True)

In [ ]:
from datasets import load_dataset
ds = load_dataset("trivia_qa", "rc.nocontext")
ds["train"].to_parquet("data/TriviaQA/rc.nocontext/triviaqa_train.parquet")
ds['train'][0]

In [ ]:
!{sys.executable} -m pip install google-genai

In [ ]:
import os
import json
import re
import string
import time
from typing import List, Set
import os
from google import genai
from google.genai import types

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm
from datasets import load_dataset


# ============================================================
# CONFIG — edit this block before running
# ============================================================

MODEL_PATH     = "meta-llama/Llama-3.2-1B-Instruct"  # any HF model path or repo id
OUTPUT_PATH    = "data/consistency_samples.jsonl"

# Sampling
SAMPLE_NUM     = 10       # how many responses to generate per question
MAX_SAMPLES    = 1000     # set to None to process the full dataset
MAX_NEW_TOKENS = 50
TEMPERATURE    = 1.0
TOP_P          = 0.9
TOP_K          = 50
DTYPE          = "float16"   # "float16" | "bfloat16" | "float32"

# Judge
JUDGE_TYPE     = "llm"                      # "rule" or "llm"
api_key = '' # GEMINI_API_KEY only needed when JUDGE_TYPE = "llm"
JUDGE_MODEL    = "llama-3.3-70b-versatile"  # groq model name
JUDGE_BATCH_SIZE = 20   # questions per API call — tune this (5–10 is safe)

# ============================================================
# Utilities
# ============================================================

def normalize_answer(s: str) -> str:
    """Lowercase, strip articles/punctuation, collapse whitespace."""
    def remove_articles(text):
        return re.sub(r'\b(a|an|the)\b', ' ', text)

    def handle_punc(text):
        exclude = set(string.punctuation + "''´`")
        return ''.join(ch if ch not in exclude else ' ' for ch in text)

    if not s:
        return ""
    return ' '.join(
        remove_articles(handle_punc(str(s).lower().replace('_', ' '))).split()
    ).strip()


def load_existing_qids(path: str) -> Set[str]:
    """Resume support — collect already-processed question IDs."""
    if not os.path.exists(path):
        return set()
    qids = set()
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                data = json.loads(line)
                qids.update(data.keys())
            except Exception:
                continue
    return qids


DTYPE_MAP = {
    "float16":  torch.float16,
    "bfloat16": torch.bfloat16,
    "float32":  torch.float32,
}


# ============================================================
# HuggingFace Sampler
# ============================================================

class HFSampler:
    """
    Wraps a HuggingFace CausalLM for batched chat-style sampling.

    The key trick: tokenize the prompt once, tile it n times along the
    batch dimension, then call generate() a single time.  This gives n
    independent samples in one forward pass instead of n separate calls.
    apply_chat_template handles model-specific tokens automatically
    (Llama-3, Mistral, Gemma, Qwen, etc. all work without changes).
    """

    def __init__(self):
        self.max_new_tokens = MAX_NEW_TOKENS
        self.temperature    = TEMPERATURE
        self.top_p          = TOP_P
        self.top_k          = TOP_K

        print(f"Loading tokenizer: {MODEL_PATH}")
        self.tokenizer = AutoTokenizer.from_pretrained(
            MODEL_PATH,
            trust_remote_code=True
        )

        # Ensure Llama 3 specific BOS/EOS tokens are set on the tokenizer if they are None
        if self.tokenizer.bos_token is None:
            self.tokenizer.bos_token = "<|begin_of_text|>"
            # If the token is not in the vocabulary, add it
            if self.tokenizer.convert_tokens_to_ids(self.tokenizer.bos_token) == self.tokenizer.unk_token_id:
                self.tokenizer.add_special_tokens({'bos_token': self.tokenizer.bos_token})

        if self.tokenizer.eos_token is None:
            self.tokenizer.eos_token = "<|end_of_text|>"
            # If the token is not in the vocabulary, add it
            if self.tokenizer.convert_tokens_to_ids(self.tokenizer.eos_token) == self.tokenizer.unk_token_id:
                self.tokenizer.add_special_tokens({'eos_token': self.tokenizer.eos_token})


        # Most causal LMs have no pad token — reuse eos so batching works
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        print(f"Loading model in {DTYPE} ...")
        self.model = AutoModelForCausalLM.from_pretrained(
            MODEL_PATH,
            torch_dtype=DTYPE_MAP[DTYPE],
            device_map="auto",        # spreads across all available GPUs
            trust_remote_code=True,
        )
        self.model.eval()
        print("Model ready.\n")

    @torch.no_grad()
    def sample(self, messages: List[dict], n: int) -> List[str]:
        """
        Args:
            messages:  chat-format list, e.g. [{"role": "user", "content": "..."}]
            n:         number of independent completions to generate

        Returns:
            list of n decoded strings (prompt tokens stripped off)
        """
        prompt = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        enc = self.tokenizer(prompt, return_tensors="pt", padding=True)
        enc = {k: v.to(self.model.device) for k, v in enc.items()}

        # Tile prompt n times → single batched generate() call
        input_ids      = enc["input_ids"].repeat(n, 1)       # [n, seq_len]
        attention_mask = enc["attention_mask"].repeat(n, 1)   # [n, seq_len]
        prompt_len     = input_ids.shape[1]

        outputs = self.model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=self.max_new_tokens,
            do_sample=True,
            temperature=self.temperature,
            top_p=self.top_p,
            top_k=self.top_k,
            pad_token_id=self.tokenizer.pad_token_id,
        )

        # Strip the prompt tokens, decode only the new generation
        new_tokens = outputs[:, prompt_len:]   # [n, new_seq_len]
        decoded    = self.tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
        return [r.strip() for r in decoded]


# ============================================================
# Groq Judge
# ============================================================



import os
from google import genai
from google.genai import types




class GroqJudge:
    """
    Sends a question + model response to a Groq-hosted LLM and asks
    whether the response correctly answers the question.

    Groq's client follows the same chat.completions.create() pattern
    as OpenAI, so the code is almost identical to the original script —
    just a different client class and model name.
    """

    BATCH_PROMPT_HEADER = (
        "For each item below, judge whether the model response correctly "
        "answers the question given the correct answers.\n"
        "Reply with ONLY a comma-separated list of 't' or 'f', one per item, "
        "in the same order. Nothing else.\n"
        "Example output for 3 items: t,f,t\n\n"
    )

    BATCH_ITEM_TEMPLATE = (
        "[{idx}] Question: {question}\n"
        "     Response: {response}\n"
        "     Correct Answers: {answers}\n"
    )

    def __init__(self):
        self.client = genai.Client(api_key=api_key)
        self._model = "gemini-3.1-flash-lite-preview"

    def judge_batch(self, items: List[dict], retries: int = 5) -> List[str]:
        """
        items: list of dicts, each with keys: question, response, answers
        Returns: list of "true"/"false"/"error" in the same order
        """
        # Build one prompt containing all items
        prompt = self.BATCH_PROMPT_HEADER
        for i, item in enumerate(items):
            prompt += self.BATCH_ITEM_TEMPLATE.format(
                idx=i + 1,
                question=item["question"],
                response=item["response"],
                answers=item["answers"],
            )

        for attempt in range(retries):
            try:
                contents = [
                    types.Content(
                        role="user",
                        parts=[types.Part.from_text(text=prompt)],
                    )
                ]
                generate_content_config = types.GenerateContentConfig(
                    thinking_config=types.ThinkingConfig(thinking_level="MINIMAL"),
                )

                raw = ""
                for chunk in self.client.models.generate_content_stream(
                    model=self._model,
                    contents=contents,
                    config=generate_content_config,
                ):
                    if chunk.text:
                        raw += chunk.text

                # Parse "t,f,t,t,f" → ["true","false","true","true","false"]
                verdicts = self._parse_batch_response(raw.strip(), len(items))
                if verdicts is not None:
                    return verdicts

            except Exception as e:
                wait = int(self._extract_retry_delay(e)) if "retryDelay" in str(e) else 2 ** attempt
                print(f"  [judge] attempt {attempt + 1} failed: {e}")
                print(f"  [judge] waiting {wait}s before retry...")
                time.sleep(wait)

        return ["error"] * len(items)

    def _parse_batch_response(self, raw: str, expected: int) -> List[str] | None:
        """Parse 't,f,t' style response. Returns None if malformed."""
        # Strip any accidental whitespace/newlines between commas
        tokens = [t.strip().lower() for t in raw.split(",")]
        if len(tokens) != expected:
            return None
        results = []
        for t in tokens:
            if t == 't' or t.startswith('t'):
                results.append("true")
            elif t == 'f' or t.startswith('f'):
                results.append("false")
            else:
                return None  # unexpected token — retry the whole call
        return results

    def _extract_retry_delay(self, e: Exception) -> float:
        """Pull the retryDelay seconds out of the 429 error string."""
        import re
        match = re.search(r"retryDelay.*?(\d+)s", str(e))
        return float(match.group(1)) if match else 4.0


# ============================================================
# ConsistencySampler
# ============================================================

class ConsistencySampler:
    """
    Main pipeline class.  For each TriviaQA question:

      1. Sample SAMPLE_NUM responses from the HF model.
      2. Judge each response (rule-based string match or Groq LLM).
      3. Keep only questions where every sample is correct (all-correct)
         or every sample is wrong (all-incorrect) — these give the cleanest
         hallucination signal for downstream H-Neuron detection.
      4. Append results to OUTPUT_PATH as newline-delimited JSON.

    If OUTPUT_PATH already exists, previously processed question IDs are
    skipped automatically so you can resume a crashed run.
    """

    UNCERTAIN_TERMS = [
        "don't know", "cannot", "not provided",
        "no information", "i'm not sure", "unclear",
    ]

    def __init__(self):
        self.sampler    = HFSampler()
        self.groq_judge = GroqJudge() if JUDGE_TYPE == "llm" else None

    # ----------------------------------------------------------
    # Judge helpers
    # ----------------------------------------------------------


    def _get_unique_responses_to_judge(self, question, responses, raw_aliases, norm_gts):
        """
        Returns a dict: {response_str -> "true"/"false"/"uncertain"}
        for all responses that need LLM judging, after rule/uncertain pre-filter.
        Values are pre-filled for uncertain ones; the rest need API calls.
        """
        result = {}
        needs_llm = []

        for resp in responses:
            if resp in result:
                continue
            if self._is_uncertain(resp):
                result[resp] = "uncertain"
            elif JUDGE_TYPE == "rule":
                result[resp] = self._rule_judge(resp, norm_gts)
            else:
                needs_llm.append(resp)   # collect unique responses needing LLM

        return result, needs_llm

    def _rule_judge(self, response: str, norm_gts: List[str]) -> str:
        norm_res = normalize_answer(response)
        for gt in norm_gts:
            if gt and gt in norm_res:
                return "true"
        return "false"

    def _is_uncertain(self, response: str) -> bool:
        return any(term in response.lower() for term in self.UNCERTAIN_TERMS)

    # ----------------------------------------------------------
    # Dataset loading
    # ----------------------------------------------------------

    def _load_dataset(self):
        """
        Load TriviaQA directly from the HuggingFace datasets library —
        same approach the user already uses, no parquet file needed.
        """
        print("Loading TriviaQA from HuggingFace datasets ...")
        ds = load_dataset("trivia_qa", "rc.nocontext", split="train")
        if MAX_SAMPLES is not None:
            ds = ds.select(range(MAX_SAMPLES))
        print(f"Loaded {len(ds)} questions.\n")
        return ds

    # ----------------------------------------------------------
    # Main loop
    # ----------------------------------------------------------

    def process_data(self):
        dataset         = self._load_dataset()
        processed_qids  = load_existing_qids(OUTPUT_PATH)
        all_correct_cnt = 0
        all_wrong_cnt   = 0
        suffix = "Respond with the answer only, without any explanation."

        os.makedirs(os.path.dirname(OUTPUT_PATH) or ".", exist_ok=True)

        # Buffer holds fully sampled questions waiting for LLM judging
        # Each entry: {qid, question, responses, raw_aliases, norm_gts}
        buffer = []

        def flush_buffer(buf):
            """Judge all buffered questions in as few API calls as possible."""
            nonlocal all_correct_cnt, all_wrong_cnt

            if not buf:
                return

            with open(OUTPUT_PATH, 'a', encoding='utf-8') as f:
                # --- Collect all (question, unique_response, answers) pairs ---
                # We need to call judge_batch on unique (question, response) pairs
                # across all buffered questions to maximize batching.
                all_items   = []   # list of {buf_idx, response} to judge
                cache_map   = [{} for _ in buf]  # per-question verdict cache

                for buf_idx, entry in enumerate(buf):
                    _, needs_llm = self._get_unique_responses_to_judge(
                        entry["question"],
                        entry["responses"],
                        entry["raw_aliases"],
                        entry["norm_gts"],
                    )
                    # Pre-fill uncertain/rule verdicts into cache
                    cache_map[buf_idx], needs_llm = self._get_unique_responses_to_judge(
                        entry["question"], entry["responses"],
                        entry["raw_aliases"], entry["norm_gts"],
                    )
                    for resp in needs_llm:
                        all_items.append({
                            "buf_idx":  buf_idx,
                            "response": resp,
                            "question": entry["question"],
                            "answers":  entry["raw_aliases"],
                        })

                # --- Call LLM in chunks of JUDGE_BATCH_SIZE ---
                if JUDGE_TYPE == "llm" and all_items:
                    for chunk_start in range(0, len(all_items), JUDGE_BATCH_SIZE):
                        chunk = all_items[chunk_start : chunk_start + JUDGE_BATCH_SIZE]
                        verdicts = self.groq_judge.judge_batch(chunk)
                        for item, verdict in zip(chunk, verdicts):
                            cache_map[item["buf_idx"]][item["response"]] = verdict

                # --- Resolve final judges list per question and write ---
                for buf_idx, entry in enumerate(buf):
                    cache   = cache_map[buf_idx]
                    judges  = [cache.get(r, "error") for r in entry["responses"]]

                    true_count = judges.count("true")
                    nonlocal_update(all_correct_cnt, all_wrong_cnt, true_count)

                    record = {
                        entry["qid"]: {
                            "question":     entry["question"],
                            "responses":    entry["responses"],
                            "judges":       judges,
                            "ground_truth": list(set(entry["raw_aliases"])),
                        }
                    }
                    f.write(json.dumps(record, ensure_ascii=False) + '\n')
                    processed_qids.add(entry["qid"])

        # nonlocal helper (Python can't reassign nonlocal ints directly in nested fn)
        counts = {"correct": 0, "wrong": 0}

        def nonlocal_update(c, w, true_count):
            if true_count == SAMPLE_NUM:
                counts["correct"] += 1
            elif true_count == 0:
                counts["wrong"] += 1

        with tqdm(total=len(dataset), desc="Processing questions") as pbar:
            for item in dataset:
                pbar.update(1)

                qid = str(item.get('question_id', ''))
                if qid in processed_qids:
                    continue

                question = item.get('question', '').strip()
                if not question or 'answer' not in item:
                    continue

                raw_aliases: List[str] = []
                for col in ['aliases', 'normalized_aliases']:
                    val = item['answer'].get(col)
                    if val:
                        raw_aliases.extend(val) if isinstance(val, list) else raw_aliases.append(str(val))

                norm_gts = [normalize_answer(a) for a in set(raw_aliases) if a]
                if not norm_gts:
                    continue

                messages = [{"role": "user", "content": f"{question} {suffix}"}]
                try:
                    responses = self.sampler.sample(messages, n=SAMPLE_NUM)
                except Exception as e:
                    tqdm.write(f"  [sampling error] qid={qid}: {e}")
                    continue

                if len(responses) < SAMPLE_NUM:
                    continue

                buffer.append({
                    "qid": qid, "question": f"{question} {suffix}",
                    "responses": responses, "raw_aliases": raw_aliases,
                    "norm_gts": norm_gts,
                })

                # Flush when buffer is full
                if len(buffer) >= JUDGE_BATCH_SIZE:
                    flush_buffer(buffer)
                    buffer.clear()
                    tqdm.write(f"  Stats → correct: {counts['correct']}, wrong: {counts['wrong']}")

            # Flush any remaining questions
            flush_buffer(buffer)

        print(f"\nDone.  all-correct: {counts['correct']},  all-incorrect: {counts['wrong']}")


# ============================================================
# Run
# ============================================================

sampler = ConsistencySampler()
sampler.process_data()

In [ ]:
import json
import os

INPUT_PATH  = "data/consistency_samples (1).jsonl"
OUTPUT_PATH = "data/answer_tokens.jsonl"

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

with open(INPUT_PATH, 'r', encoding='utf-8') as fin, \
     open(OUTPUT_PATH, 'w', encoding='utf-8') as fout:

    for line in fin:
        line = line.strip()
        if not line:
            continue

        raw = json.loads(line)

        # Each line is {"tc_1": {...}} — one qid per line
        for qid, data in raw.items():
            question     = data["question"]
            responses    = data["responses"]
            judges       = data["judges"]
            ground_truth = data["ground_truth"]

            # One output record per (question, response) pair
            # answer_tokens: [] tells extract_activations.py to use
            # all output tokens (the --locations output path)
            for response, judge in zip(responses, judges):
                record = {
                    "qid":          qid,
                    "question":     question,
                    "response":     response,
                    "label":        1 if judge == "false" else 0,
                    "answer_tokens": []   # empty = use all output tokens
                }
                fout.write(json.dumps(record, ensure_ascii=False) + '\n')

print("Done.")

# Sanity check — print first 3 records
with open(OUTPUT_PATH) as f:
    for i, line in enumerate(f):
        if i >= 3: break
        print(json.loads(line))

Done.
{'qid': 'tc_1', 'question': 'Which American-born Sinclair won the Nobel Prize for Literature in 1930? Respond with the answer only, without any explanation.', 'response': 'Dorothy Johnson', 'label': 1, 'answer_tokens': []}
{'qid': 'tc_1', 'question': 'Which American-born Sinclair won the Nobel Prize for Literature in 1930? Respond with the answer only, without any explanation.', 'response': 'A. S. Byatt', 'label': 1, 'answer_tokens': []}
{'qid': 'tc_1', 'question': 'Which American-born Sinclair won the Nobel Prize for Literature in 1930? Respond with the answer only, without any explanation.', 'response': 'Orwell', 'label': 1, 'answer_tokens': []}


In [ ]:
import json
import torch
from pathlib import Path
from tqdm import tqdm

# ============================================================
# CONFIG
# ============================================================

ANSWER_TOKENS_PATH = "data/answer_tokens.jsonl"
OUTPUT_DIR         = "data/activations"
METHOD             = "mean"   # "mean" or "max"

# ============================================================
# Make sure model + tokenizer are already loaded from previous cell
# and ActivationExtractor, get_neuron_activations, compute_cett
# are already defined
# ============================================================

def extract_dataset_activations(jsonl_path, output_dir, model, tokenizer, extractor):
    """
    Reads answer_tokens.jsonl — each line:
      {
        "qid":           "tc_1",
        "question":      "Which American...",   ← this is the prompt
        "response":      "Sinclair Lewis",
        "label":         0 or 1,
        "answer_tokens": []                     ← empty = use all response tokens
      }

    Saves:
      output_dir/activations.pt  →  {features, labels, qids}
    """
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    features, labels, qids = [], [], []
    errors = 0

    with open(jsonl_path) as f:
        lines = f.readlines()

    for line in tqdm(lines, desc="Extracting activations"):
        line = line.strip()
        if not line:
            continue

        sample = json.loads(line)

        # ---- field name fix: your file uses "question" not "prompt" ----
        prompt   = sample.get("question") or sample.get("prompt", "")
        response = sample.get("response", "")
        label    = sample.get("label", -1)
        qid      = sample.get("qid", "")

        # answer_tokens: [] means use all response tokens (our case)
        raw_indices = sample.get("answer_tokens") or None
        token_indices = raw_indices if raw_indices else None

        try:
            layer_acts  = get_neuron_activations(
                extractor, tokenizer, prompt, response,
                token_indices=token_indices
            )
            feature_vec = compute_cett(layer_acts, method=METHOD)

            features.append(feature_vec)
            labels.append(label)
            qids.append(qid)

        except Exception as e:
            tqdm.write(f"  [error] qid={qid}: {e}")
            errors += 1
            continue

    if not features:
        print("No features extracted. Check your input file.")
        return None, None

    features_tensor = torch.stack(features)   # [N, num_layers * intermediate_size]
    labels_tensor   = torch.tensor(labels)

    save_path = f"{output_dir}/activations.pt"
    torch.save({
        "features": features_tensor,
        "labels":   labels_tensor,
        "qids":     qids
    }, save_path)

    print(f"\nSaved → {save_path}")
    print(f"  Samples  : {len(features_tensor)}")
    print(f"  Feature dim : {features_tensor.shape[1]}")
    print(f"  Label dist  : {labels_tensor.sum().item()} hallucinated, "
          f"{(labels_tensor == 0).sum().item()} faithful")
    print(f"  Errors skipped: {errors}")

    return features_tensor, labels_tensor


# ============================================================
# Run
# ============================================================

extractor = ActivationExtractor(model)

features, labels = extract_dataset_activations(
    jsonl_path  = ANSWER_TOKENS_PATH,
    output_dir  = OUTPUT_DIR,
    model       = model,
    tokenizer   = tokenizer,
    extractor   = extractor,
)

In [12]:
import json
import random
from pathlib import Path
from collections import defaultdict

# ============================================================
# CONFIG
# ============================================================

ANSWER_TOKENS_PATH = "data/answer_tokens.jsonl"
OUTPUT_PATH        = "data/train_qids.json"

# How many samples PER CLASS to select (total = 2 * NUM_SAMPLES_PER_CLASS)
# The repo default is 1000 total → 500 per class
NUM_SAMPLES_PER_CLASS = 500

RANDOM_SEED = 42

# ============================================================

def sample_balanced_ids(jsonl_path, output_path,
                         num_per_class=500, seed=42):
    """
    Reads answer_tokens.jsonl and groups qids by label.
    Then samples num_per_class from each label bucket,
    producing a balanced train/test split.

    Why balance matters:
      If your data is 80% hallucinated, a classifier that always predicts
      "hallucinated" gets 80% accuracy for free — it learns nothing.
      Equal class sizes force it to actually discriminate.

    Output JSON structure:
      {
        "train": ["tc_1", "tc_8", ...],   ← balanced mix of 0s and 1s
        "test":  ["tc_3", "tc_5", ...]    ← held-out balanced mix
      }
    """
    random.seed(seed)

    # Group qids by label — use set() to deduplicate
    # (same qid can appear multiple times if you have multiple responses)
    buckets = defaultdict(set)   # label → set of qids

    with open(jsonl_path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            sample = json.loads(line)
            label  = sample.get("label", -1)
            qid    = sample.get("qid", "")
            if label in (0, 1) and qid:
                buckets[label].add(qid)

    faithful_ids      = list(buckets[0])   # label 0 = correct
    hallucinated_ids  = list(buckets[1])   # label 1 = hallucinated

    print(f"Available qids — faithful: {len(faithful_ids)}, "
          f"hallucinated: {len(hallucinated_ids)}")

    # Cap at what's available
    n = min(num_per_class, len(faithful_ids), len(hallucinated_ids))
    if n < num_per_class:
        print(f"  Warning: only {n} samples available per class "
              f"(requested {num_per_class}). Using {n}.")

    # Split each class 80/20 into train/test
    train_n = int(n * 0.8)
    test_n  = n - train_n

    random.shuffle(faithful_ids)
    random.shuffle(hallucinated_ids)

    train_ids = faithful_ids[:train_n]     + hallucinated_ids[:train_n]
    test_ids  = faithful_ids[train_n:n]    + hallucinated_ids[train_n:n]

    # Shuffle so train_ids isn't [all 0s, then all 1s]
    random.shuffle(train_ids)
    random.shuffle(test_ids)

    result = {
        "train": train_ids,
        "test":  test_ids,
    }

    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, 'w') as f:
        json.dump(result, f, indent=2)

    print(f"\nSaved → {output_path}")
    print(f"  Train : {len(train_ids)} qids  ({train_n} per class)")
    print(f"  Test  : {len(test_ids)} qids  ({test_n} per class)")

    return result


# ============================================================
# Run
# ============================================================

split = sample_balanced_ids(
    jsonl_path    = ANSWER_TOKENS_PATH,
    output_path   = OUTPUT_PATH,
    num_per_class = NUM_SAMPLES_PER_CLASS,
    seed          = RANDOM_SEED,
)

print("\nSample train IDs:", split["train"][:5])
print("Sample test  IDs:", split["test"][:5])

Available qids — faithful: 514, hallucinated: 935

Saved → data/train_qids.json
  Train : 800 qids  (400 per class)
  Test  : 200 qids  (100 per class)

Sample train IDs: ['tc_597', 'tc_1317', 'tc_578', 'tc_1633', 'tc_860']
Sample test  IDs: ['tc_1168', 'tc_300', 'tc_207', 'tc_1013', 'tc_1102']


In [ ]:
# save every 500 samples in case Colab crashes
if len(features) % 500 == 0:
    torch.save({"features": torch.stack(features), "labels": torch.tensor(labels), "qids": qids},
               f"{OUTPUT_DIR}/activations_checkpoint.pt")

In [ ]:
import gc
import torch

# Free the LLM from GPU/CPU memory — you don't need it anymore
# after activations are extracted
if 'model' in dir():
    del model
if 'extractor' in dir():
    del extractor

gc.collect()
torch.cuda.empty_cache()
print("Memory cleared.")

In [ ]:
import json
import os
import joblib
import numpy as np
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                              roc_auc_score, classification_report)
from pathlib import Path

# ============================================================
# CONFIG
# ============================================================

ACTIVATIONS_PATH = "/content/drive/MyDrive/Colab_Output/activations.pt"
TRAIN_QIDS_PATH  = "data/train_qids.json"
SAVE_MODEL_PATH  = "models/detector.pkl"

TRAIN_MODE = "1-vs-1"   # "1-vs-1" for best accuracy (detector use case)
                         # "3-vs-1" to isolate H-Neurons specifically

PENALTY    = "l2"        # "l2" → higher accuracy, use this for a detector
                         # "l1" → sparse weights, use this to find H-Neurons
C          = 1.0         # inverse regularization strength
                         # smaller = stronger regularization
                         # only matters for l1 (controls how many neurons survive)

# ============================================================
# Step 1 — Load activations.pt and split by train/test qids
# ============================================================

def load_and_split(activations_path, train_qids_path):
    """
    Your activations.pt has:
        features : [N, num_layers * intermediate_size]  float tensors
        labels   : [N]  — 1 = hallucinated, 0 = faithful
        qids     : list of N qid strings (e.g. "tc_1", "tc_8")

    train_qids.json has:
        {"train": ["tc_1", ...], "test": ["tc_3", ...]}

    One qid can appear multiple times in features (once per response),
    so we collect ALL rows whose qid is in the train/test set.
    """
    print("Loading activations ...")
    checkpoint = torch.load(activations_path, map_location="cpu")

    # all_features = checkpoint["features"].to(torch.float32).numpy()
    all_features = checkpoint["features"].to(torch.float16).numpy()
    # stays float16 — only converted to float32 in chunks inside train_classifier
    all_labels   = checkpoint["labels"].numpy().astype(int)
    all_qids     = checkpoint["qids"]   # list of strings

    print(f"  Total records : {len(all_qids)}")
    print(f"  Feature dim   : {all_features.shape[1]}")
    print(f"  Hallucinated  : {all_labels.sum()}  |  Faithful: {(all_labels == 0).sum()}")

    with open(train_qids_path) as f:
        split = json.load(f)

    train_set = set(split["train"])
    test_set  = set(split["test"])

    train_mask = np.array([q in train_set for q in all_qids])
    test_mask  = np.array([q in test_set  for q in all_qids])

    X_train = all_features[train_mask]
    y_train = all_labels[train_mask]
    X_test  = all_features[test_mask]
    y_test  = all_labels[test_mask]

    print(f"\nSplit:")
    print(f"  Train — {len(X_train)} records  "
          f"({y_train.sum()} hallucinated, {(y_train==0).sum()} faithful)")
    print(f"  Test  — {len(X_test)} records  "
          f"({y_test.sum()} hallucinated, {(y_test==0).sum()} faithful)")

    return X_train, y_train, X_test, y_test, all_qids


# ============================================================
# Step 2 — Build X, y for the chosen train_mode
# ============================================================

def build_training_data(X_train, y_train, train_mode):
    """
    1-vs-1: hallucinated answer tokens (label=1)
            vs faithful answer tokens (label=0)
            → use X_train / y_train directly

    3-vs-1: hallucinated answer tokens (label=1)
            vs faithful answer tokens + ALL other-token rows (label=0)

    Since we're using full output tokens (answer_tokens=[]),
    every row already represents the full response activation.
    In 3-vs-1 the negative class is simply everything that isn't
    a hallucinated response — which is already captured by label=0.
    So both modes use the same data here; the distinction matters
    more when you have separate answer_token vs other_token activation
    files (the repo's original setup). We preserve the flag for clarity.
    """
    if train_mode == "1-vs-1":
        print(f"\nMode 1-vs-1: {(y_train==1).sum()} positives vs "
              f"{(y_train==0).sum()} negatives")
        return X_train, y_train

    elif train_mode == "3-vs-1":
        # In our setup all output tokens are already included per record,
        # so label=0 already combines faithful answers + non-answer context.
        # Nothing extra to add — same matrix, documented intent differs.
        print(f"\nMode 3-vs-1: {(y_train==1).sum()} hallucinated vs "
              f"{(y_train==0).sum()} faithful/other")
        return X_train, y_train


# ============================================================
# Step 3 — Train
# ============================================================

def train_classifier(X, y, penalty, C, solver=None):
    """
    Memory-efficient training:
    - Keep X as float16 numpy until the last moment
    - Fit scaler in one pass, transform in chunks to avoid a full float32 copy
    - Use saga solver which supports sparse updates and lower memory than liblinear
    """
    if solver is None:
        solver = "saga"   # saga works better than liblinear on memory

    print(f"\nFitting scaler ...")
    scaler = StandardScaler()

    # Fit scaler on float16 — StandardScaler only needs mean/std,
    # which are computed correctly even from float16 input
    scaler.fit(X)

    # Transform in chunks to avoid materializing a full float32 copy at once
    # Each chunk is ~50MB instead of the full 5GB
    CHUNK = 500
    X_scaled_parts = []
    for start in range(0, len(X), CHUNK):
        chunk = X[start : start + CHUNK].astype(np.float32)  # float16 → float32 per chunk
        X_scaled_parts.append(scaler.transform(chunk))
        del chunk

    X_scaled = np.vstack(X_scaled_parts)
    del X_scaled_parts

    print(f"Training LogisticRegression  penalty={penalty}  C={C}  solver={solver} ...")
    clf = LogisticRegression(
        penalty=penalty,
        C=C,
        solver=solver,
        max_iter=1000,
        random_state=42,
        class_weight="balanced",
        verbose=1,
    )
    clf.fit(X_scaled, y)
    del X_scaled   # free immediately after fit

    coef = clf.coef_[0]
    nonzero = (coef != 0).sum()
    total   = len(coef)
    if penalty == "l1":
        print(f"\nH-Neurons found: {nonzero} / {total} ({100 * nonzero / total:.3f}%)")
    else:
        print(f"\nNon-zero weights: {nonzero} / {total}")

    return clf, scaler


# ============================================================
# Step 4 — Evaluate
# ============================================================

def evaluate(clf, scaler, X, y, split_name="Test"):
    X_scaled = scaler.transform(X)
    preds    = clf.predict(X_scaled)
    probs    = clf.predict_proba(X_scaled)[:, 1]

    acc             = accuracy_score(y, preds)
    prec, rec, f1, _= precision_recall_fscore_support(y, preds, average="binary")
    auroc           = roc_auc_score(y, probs)

    print(f"\n{'='*40}")
    print(f"  {split_name} Results")
    print(f"{'='*40}")
    print(f"  Accuracy  : {acc:.4f}")
    print(f"  Precision : {prec:.4f}")
    print(f"  Recall    : {rec:.4f}")
    print(f"  F1        : {f1:.4f}")
    print(f"  AUROC     : {auroc:.4f}")
    print(f"\n{classification_report(y, preds, target_names=['faithful','hallucinated'])}")


# ============================================================
# Step 5 — Inspect top H-Neurons (useful for both l1 and l2)
# ============================================================

def inspect_h_neurons(clf, top_n=20):
    """
    Decode the flat weight index back to (layer, neuron_within_layer).
    For l1: only non-zero weights are H-Neurons.
    For l2: shows the top_n highest-magnitude weights.
    """
    coef = clf.coef_[0]

    if PENALTY == "l1":
        indices = np.where(coef != 0)[0]
    else:
        indices = np.argsort(np.abs(coef))[::-1][:top_n]

    # We need intermediate_size to decode layer + position
    # Llama-3.2-1B: 16 layers, intermediate_size = 8192
    # This is loaded from the model config automatically in the next line
    try:
        from transformers import AutoConfig
        cfg = AutoConfig.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")
        intermediate_size = cfg.intermediate_size
        num_layers        = cfg.num_hidden_layers
    except Exception:
        # Fallback if config unavailable
        intermediate_size = coef.shape[0] // 16
        num_layers        = 16

    print(f"\nTop H-Neurons  (intermediate_size={intermediate_size}, "
          f"num_layers={num_layers}):")
    print(f"  {'rank':<5} {'layer':<7} {'neuron':<8} {'weight':>10}")
    print(f"  {'-'*35}")

    sorted_indices = indices[np.argsort(np.abs(coef[indices]))[::-1]]
    for rank, flat_idx in enumerate(sorted_indices[:top_n]):
        layer  = int(flat_idx) // intermediate_size
        neuron = int(flat_idx) %  intermediate_size
        weight = coef[flat_idx]
        print(f"  {rank+1:<5} {layer:<7} {neuron:<8} {weight:>+10.4f}")


# ============================================================
# Run
# ============================================================

X_train, y_train, X_test, y_test, _ = load_and_split(
    ACTIVATIONS_PATH, TRAIN_QIDS_PATH
)

X_tr, y_tr = build_training_data(X_train, y_train, TRAIN_MODE)

clf, scaler = train_classifier(X_tr, y_tr, penalty=PENALTY, C=C)

evaluate(clf, scaler, X_tr, y_tr, split_name="Train")
evaluate(clf, scaler, X_test, y_test, split_name="Test")

inspect_h_neurons(clf, top_n=20)

# Save
Path(SAVE_MODEL_PATH).parent.mkdir(parents=True, exist_ok=True)
joblib.dump({"clf": clf, "scaler": scaler}, SAVE_MODEL_PATH)
print(f"\nModel saved → {SAVE_MODEL_PATH}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import shutil
import os

source_path = 'data/activations/activations.pt'
destination_dir = '/content/drive/MyDrive/Colab_Output'

# Create the destination directory if it doesn't exist
os.makedirs(destination_dir, exist_ok=True)

# Move the file
shutil.move(source_path, os.path.join(destination_dir, 'activations.pt'))

print(f"'activations.pt' moved from '{source_path}' to '{destination_dir}/activations.pt'")

In [ ]:
import json
import gc
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
from pathlib import Path

# ============================================================
# CONFIG
# ============================================================

ACTIVATIONS_PATH = "data/activations/activations.pt"
TRAIN_QIDS_PATH  = "data/train_qids.json"
SAVE_MODEL_PATH  = "models/detector_pytorch.pt"

PENALTY     = "l2"    # "l1" → find H-Neurons  |  "l2" → best accuracy
LAMBDA      = 1e-4    # regularization strength (bigger = more regularization)
                      # for l1: controls sparsity (try 1e-3 to 1e-5)
                      # for l2: controls overfitting (try 1e-3 to 1e-5)

LR          = 1e-3    # learning rate
EPOCHS      = 15      # stop early if val loss stops improving
BATCH_SIZE  = 256
PATIENCE    = 5       # stop if val loss doesn't improve for this many epochs

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


# ============================================================
# Step 1 — Load + split (same as before, reuse if already loaded)
# ============================================================

def load_and_split(activations_path, train_qids_path):
    print("Loading activations ...")
    ckpt = torch.load(activations_path, map_location="cpu")

    # Keep as float16 tensor on CPU until we push batches to GPU
    # This is the key memory saving — 131k * 10000 * 2 bytes = ~2.5GB
    all_features = ckpt["features"].to(torch.float16)
    all_labels   = ckpt["labels"].long()
    all_qids     = ckpt["qids"]

    print(f"  Records     : {len(all_qids)}")
    print(f"  Feature dim : {all_features.shape[1]}")
    print(f"  Hallucinated: {all_labels.sum().item()}  "
          f"Faithful: {(all_labels == 0).sum().item()}")

    with open(train_qids_path) as f:
        split = json.load(f)

    train_set = set(split["train"])
    test_set  = set(split["test"])

    train_mask = torch.tensor([q in train_set for q in all_qids])
    test_mask  = torch.tensor([q in test_set  for q in all_qids])

    X_train = all_features[train_mask]
    y_train = all_labels[train_mask]
    X_test  = all_features[test_mask]
    y_test  = all_labels[test_mask]

    print(f"\n  Train: {len(X_train)}  "
          f"({y_train.sum().item()} hallucinated, {(y_train==0).sum().item()} faithful)")
    print(f"  Test : {len(X_test)}   "
          f"({y_test.sum().item()} hallucinated, {(y_test==0).sum().item()} faithful)")

    del ckpt
    gc.collect()

    return X_train, y_train, X_test, y_test


# ============================================================
# Step 2 — Online standardization layer
# ============================================================

class OnlineStandardScaler(nn.Module):
    """
    Computes mean and std from data once, stores as buffers (not parameters),
    and applies z-score normalization in the forward pass on GPU.

    Why not sklearn StandardScaler?
      - sklearn needs the full float32 matrix in RAM at once → crashes
      - This computes stats in float32 chunks, then normalizes on GPU per batch
    """
    def __init__(self, num_features):
        super().__init__()
        # register_buffer: saved with model state but not trained
        self.register_buffer("mean_", torch.zeros(num_features))
        self.register_buffer("std_",  torch.ones(num_features))
        self.fitted = False

    def fit(self, X_fp16: torch.Tensor, chunk_size: int = 1000):
        """Compute mean/std in float32 chunks to avoid OOM."""
        print("  Fitting scaler (chunked, float32) ...")
        n, d   = X_fp16.shape
        mean   = torch.zeros(d, dtype=torch.float32)
        mean_sq = torch.zeros(d, dtype=torch.float32)

        for start in range(0, n, chunk_size):
            chunk = X_fp16[start : start + chunk_size].float()  # fp16 → fp32
            mean    += chunk.sum(0)
            mean_sq += chunk.pow(2).sum(0)
            del chunk

        mean    /= n
        mean_sq /= n
        std      = (mean_sq - mean.pow(2)).clamp(min=1e-8).sqrt()

        # Store as fp32 buffers — normalization always done in fp32
        self.mean_.copy_(mean)
        self.std_.copy_(std)
        self.fitted = True
        print("  Scaler fitted.")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x arrives as float16 from DataLoader → cast up, normalize, return fp32
        return (x.float() - self.mean_) / self.std_


# ============================================================
# Step 3 — Model: linear probe (logistic regression in PyTorch)
# ============================================================

class HallucinationProbe(nn.Module):
    """
    Single linear layer — equivalent to logistic regression.
    Keeping it linear means the weights ARE the H-Neuron scores,
    interpretable the same way as sklearn's coef_.

    If accuracy plateaus you can add a hidden layer — see commented block below.
    """
    def __init__(self, input_dim: int, scaler: OnlineStandardScaler):
        super().__init__()
        self.scaler = scaler                     # normalization baked in
        self.linear = nn.Linear(input_dim, 1)
        # xavier_uniform_ expects a 2D tensor [fan_out, fan_in] — use weight directly
        nn.init.xavier_uniform_(self.linear.weight)
        nn.init.zeros_(self.linear.bias)

        # ---- Optional: uncomment for a shallow MLP if linear underfits ----
        # self.net = nn.Sequential(
        #     nn.Linear(input_dim, 256),
        #     nn.ReLU(),
        #     nn.Dropout(0.3),
        #     nn.Linear(256, 1)
        # )

    def forward(self, x):
        x = self.scaler(x)           # normalize on GPU, output is float32
        return self.linear(x).squeeze(-1)   # [batch] logits

        # ---- if using MLP: ----
        # return self.net(x).squeeze(-1)


# ============================================================
# Step 4 — Regularization loss
# ============================================================

def reg_loss(model: HallucinationProbe, penalty: str, lam: float) -> torch.Tensor:
    """
    L1: drives weights to zero → sparse H-Neurons survive
    L2: keeps all weights small → better generalization
    Applied only to the linear weight, not the bias.
    """
    w = model.linear.weight   # shape [1, input_dim]
    if penalty == "l1":
        return lam * w.abs().sum()
    else:
        return lam * w.pow(2).sum()


# ============================================================
# Step 5 — Train loop
# ============================================================

def train(model, train_loader, val_loader, epochs, lr, penalty, lam, patience):
    model.to(DEVICE)

    # Class weights to handle imbalance — same as sklearn's class_weight="balanced"
    # Count from the full training set
    all_y = torch.cat([y for _, y in train_loader])
    n_pos = all_y.sum().float()
    n_neg = (all_y == 0).sum().float()
    pos_weight = (n_neg / n_pos).to(DEVICE)
    print(f"  Pos weight (neg/pos ratio): {pos_weight.item():.2f}")

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0)

    # Cosine LR decay — gently reduces LR toward 0 over training
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_val_loss  = float("inf")
    best_state     = None
    epochs_no_improve = 0

    print(f"\n{'Epoch':>6} {'Train Loss':>12} {'Val Loss':>10} "
          f"{'Val Acc':>9} {'Val AUC':>9}")
    print("-" * 52)

    for epoch in range(1, epochs + 1):
        # ---- Train ----
        model.train()
        train_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(DEVICE)          # fp16 tensor
            y_batch = y_batch.float().to(DEVICE)

            optimizer.zero_grad()
            logits = model(X_batch)                # scaler casts to fp32 inside
            loss   = criterion(logits, y_batch) + reg_loss(model, penalty, lam)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * len(y_batch)

        train_loss /= len(train_loader.dataset)
        scheduler.step()

        # ---- Validate ----
        model.eval()
        val_loss, all_logits, all_labels = 0.0, [], []
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch = X_batch.to(DEVICE)
                y_batch = y_batch.float().to(DEVICE)
                logits  = model(X_batch)
                loss    = criterion(logits, y_batch)
                val_loss += loss.item() * len(y_batch)
                all_logits.append(logits.cpu())
                all_labels.append(y_batch.cpu())

        val_loss   /= len(val_loader.dataset)
        all_logits  = torch.cat(all_logits)
        all_labels  = torch.cat(all_labels)
        val_preds   = (torch.sigmoid(all_logits) > 0.5).long()
        val_acc     = (val_preds == all_labels.long()).float().mean().item()

        from sklearn.metrics import roc_auc_score
        try:
            val_auc = roc_auc_score(all_labels.numpy(),
                                     torch.sigmoid(all_logits).numpy())
        except Exception:
            val_auc = float("nan")

        print(f"{epoch:>6}   {train_loss:>10.4f}   {val_loss:>8.4f}   "
              f"{val_acc:>7.4f}   {val_auc:>7.4f}")

        # ---- Early stopping ----
        if val_loss < best_val_loss - 1e-4:
            best_val_loss = val_loss
            best_state    = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"\n  Early stop at epoch {epoch} "
                      f"(no improvement for {patience} epochs)")
                break

    # Restore best weights
    model.load_state_dict(best_state)
    print(f"\n  Best val loss: {best_val_loss:.4f}")
    return model


# ============================================================
# Step 6 — Final evaluation
# ============================================================

def evaluate(model, scaler, X, y, split_name="Test"):
    from sklearn.metrics import (classification_report, roc_auc_score,
                                  accuracy_score)

    model.eval()
    dataset    = TensorDataset(X, y)
    loader     = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

    all_logits, all_labels = [], []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            logits = model(X_batch.to(DEVICE)).cpu()
            all_logits.append(logits)
            all_labels.append(y_batch)

    logits = torch.cat(all_logits)
    labels = torch.cat(all_labels)
    probs  = torch.sigmoid(logits).numpy()
    preds  = (probs > 0.5).astype(int)
    y_np   = labels.numpy()

    acc  = accuracy_score(y_np, preds)
    auroc = roc_auc_score(y_np, probs)

    print(f"\n{'='*45}")
    print(f"  {split_name}")
    print(f"{'='*45}")
    print(f"  Accuracy : {acc:.4f}   AUROC: {auroc:.4f}")
    print(classification_report(y_np, preds,
                                  target_names=["faithful", "hallucinated"]))


# ============================================================
# Step 7 — Inspect H-Neurons
# ============================================================

def inspect_h_neurons(model, top_n=20):
    coef = model.linear.weight.detach().cpu().float().numpy()[0]

    try:
        from transformers import AutoConfig
        cfg = AutoConfig.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")
        intermediate_size = cfg.intermediate_size
    except Exception:
        intermediate_size = len(coef) // 16

    if PENALTY == "l1":
        indices = np.where(np.abs(coef) > 1e-6)[0]
        print(f"\nH-Neurons (non-zero): {len(indices)} / {len(coef)} "
              f"({100*len(indices)/len(coef):.3f}%)")
    else:
        indices = np.argsort(np.abs(coef))[::-1][:top_n]
        print(f"\nTop {top_n} neurons by weight magnitude:")

    sorted_idx = indices[np.argsort(np.abs(coef[indices]))[::-1]]
    print(f"  {'rank':<5} {'layer':<7} {'neuron':<8} {'weight':>10}")
    print(f"  {'-'*35}")
    for rank, flat_idx in enumerate(sorted_idx[:top_n]):
        layer  = int(flat_idx) // intermediate_size
        neuron = int(flat_idx) %  intermediate_size
        print(f"  {rank+1:<5} {layer:<7} {neuron:<8} {coef[flat_idx]:>+10.4f}")


# ============================================================
# Run
# ============================================================

X_train, y_train, X_test, y_test = load_and_split(
    ACTIVATIONS_PATH, TRAIN_QIDS_PATH
)

input_dim = X_train.shape[1]

# Fit scaler on training data only
scaler = OnlineStandardScaler(input_dim)
scaler.fit(X_train)
scaler.to(DEVICE)

# Build probe
probe = HallucinationProbe(input_dim, scaler)

# 80/20 val split from training data to monitor learning
val_size   = int(0.2 * len(X_train))
train_size = len(X_train) - val_size
train_ds, val_ds = random_split(
    TensorDataset(X_train, y_train), [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                           pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                           pin_memory=True)

probe = train(probe, train_loader, val_loader,
              epochs=EPOCHS, lr=LR,
              penalty=PENALTY, lam=LAMBDA,
              patience=PATIENCE)

evaluate(probe, scaler, X_test, y_test, split_name="Test Set")

inspect_h_neurons(probe, top_n=20)

# Save
Path(SAVE_MODEL_PATH).parent.mkdir(parents=True, exist_ok=True)
torch.save({
    "model_state": probe.state_dict(),
    "input_dim":   input_dim,
    "penalty":     PENALTY,
    "lambda":      LAMBDA,
}, SAVE_MODEL_PATH)
print(f"\nSaved → {SAVE_MODEL_PATH}")


In [ ]:
import gc
import torch

# Free the LLM from GPU/CPU memory — you don't need it anymore
gc.collect()
torch.cuda.empty_cache()
print("Memory cleared.")

In [ ]:
import os
import time
drive_path = "/content/drive/MyDrive/Colab_Output/"

# print(os.listdir(drive_path))

torch.save({
    "model_state": probe.state_dict(),
    "input_dim":   input_dim,
    "penalty":     PENALTY,
    "lambda":      LAMBDA,
}, drive_path+"detector_pytorch.pt")

time.sleep(5)

print(os.listdir(drive_path))

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig

# ============================================================
# CONFIG
# ============================================================

MODEL_PATH       = "meta-llama/Llama-3.1-8B-Instruct"
DETECTOR_PATH    = "models/detector_pytorch.pt"

# Threshold for hallucination warning  (0.5 = default, lower = more sensitive)
HALLUCINATION_THRESHOLD = 0.5

DTYPE  = "float16"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DTYPE_MAP = {
    "float16":  torch.float16,
    "bfloat16": torch.bfloat16,
    "float32":  torch.float32,
}


# ============================================================
# Step 1 — Rebuild the probe and load weights
# ============================================================

# OnlineStandardScaler and HallucinationProbe must already be defined
# from the training cell above. If running in a fresh session, re-run
# that cell first.

def load_detector(detector_path, device):
    """
    Loads the saved probe + scaler state.
    Returns the probe in eval mode on the target device.
    """
    ckpt      = torch.load(detector_path, map_location="cpu")
    input_dim = ckpt["input_dim"]

    # Rebuild scaler and probe with the same architecture
    scaler = OnlineStandardScaler(input_dim)
    probe  = HallucinationProbe(input_dim, scaler)
    probe.load_state_dict(ckpt["model_state"])
    probe.eval()
    probe.to(device)

    print(f"Detector loaded  (input_dim={input_dim}, "
          f"penalty={ckpt.get('penalty','?')})")
    return probe


# ============================================================
# Step 2 — (Optional) Inspect top H-Neuron weights for interpretability
# ============================================================

def get_h_neuron_indices(probe, model_config, top_k=None):
    """
    Maps flat weight indices to (layer, neuron) pairs.
    Positive weights → neuron fires more during hallucination (H-Neurons).
    This is for INTERPRETABILITY only — not used in inference scoring.
    """
    import numpy as np
    weights    = probe.linear.weight.detach().cpu().float().numpy()[0]    # [D]
    inter_size = (model_config.intermediate_size
                  if hasattr(model_config, "intermediate_size")
                  else model_config.text_config.intermediate_size)

    positive_flat = np.where(weights > 0)[0]

    if top_k is not None and len(positive_flat) > top_k:
        top_indices   = np.argsort(weights[positive_flat])[::-1][:top_k]
        positive_flat = positive_flat[top_indices]

    neuron_map = {}
    for flat_idx in positive_flat:
        layer_idx  = int(flat_idx) // inter_size
        neuron_idx = int(flat_idx) %  inter_size
        neuron_map.setdefault(layer_idx, []).append(neuron_idx)

    total = sum(len(v) for v in neuron_map.values())
    print(f"H-Neurons identified: {total} across {len(neuron_map)} layers")
    return neuron_map


# ============================================================
# Step 3 — Hallucination monitor
# ============================================================

class HallucinationMonitor:
    """
    Generates a response to a question and scores it for hallucination.

    Theory (H-Neurons paper):
      The model's FFN intermediate activations carry a hallucination signal.
      We compute the CETT feature vector — mean absolute activation per
      neuron across all RESPONSE tokens — then pass it to the linear probe
      trained to discriminate hallucinated vs. faithful responses.

    Implementation:
      1. Generate the response with greedy decoding (no hooks needed).
      2. Run a SINGLE forward pass on the full (prompt + response) sequence.
         This exactly replicates the training-time feature extraction.
      3. Slice to response token positions and compute CETT via compute_cett().
      4. Feed the full dense feature vector to the probe.

    Why not hook during generation?
      Autoregressive generation processes one token per step; capturing only the
      last step's activation ignores all earlier response tokens and produces a
      different distribution than the mean-over-all-tokens used in training.
      The single post-generation forward pass is both correct and cheaper
      (one pass vs. N passes for N generated tokens).
    """

    def __init__(self, llm, probe, tokenizer, threshold):
        self.llm       = llm
        self.probe     = probe
        self.tokenizer = tokenizer
        self.threshold = threshold

    @torch.no_grad()
    def generate_with_warning(self, question: str, max_new_tokens: int = 100):
        """
        Generate a response then score it for hallucination.

        Returns:
            (response_str, hallucination_probability)
        """
        suffix   = "Respond with the answer only, without any explanation."
        messages = [{"role": "user", "content": f"{question} {suffix}"}]

        prompt_str = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        enc        = self.tokenizer(prompt_str, return_tensors="pt").to(self.llm.device)
        prompt_len = enc["input_ids"].shape[1]

        # ------------------------------------------------------------------
        # Step 1: Generate the response (greedy, no hooks required here)
        # ------------------------------------------------------------------
        output_ids = self.llm.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=self.tokenizer.eos_token_id,
        )

        response_ids = output_ids[0, prompt_len:]
        response     = self.tokenizer.decode(
            response_ids, skip_special_tokens=True
        ).strip()

        if len(response_ids) == 0:
            print("(no response generated)")
            return response, 0.0

        # ------------------------------------------------------------------
        # Step 2: Single forward pass on full (prompt + response)
        # This is identical to how activations were extracted during training.
        # ActivationExtractor and compute_cett must be defined (from
        # the earlier cells in this notebook).
        # ------------------------------------------------------------------
        extractor = ActivationExtractor(self.llm)
        extractor.register_hooks()
        self.llm(output_ids)          # one complete forward pass
        extractor.remove_hooks()

        # ------------------------------------------------------------------
        # Step 3: Slice to response token positions only, then compute CETT
        # (mean absolute activation per neuron) — matches training exactly
        # ------------------------------------------------------------------
        response_token_indices = list(range(prompt_len, output_ids.shape[1]))
        sliced = {
            layer_idx: act[0, response_token_indices, :]   # [n_resp_tokens, D]
            for layer_idx, act in extractor.activations.items()
        }

        # CETT: mean |activation| across response tokens → [num_layers * D]
        feat_vec = compute_cett(sliced, method="mean")
        feat_vec = feat_vec.unsqueeze(0).to(DEVICE)        # [1, input_dim]

        # ------------------------------------------------------------------
        # Step 4: Run the probe (scaler is baked into probe.forward)
        # ------------------------------------------------------------------
        self.probe.eval()
        logit = self.probe(feat_vec)
        prob  = torch.sigmoid(logit).item()

        # Print result
        print(f"\n{'─'*55}")
        print(f"  Q: {question}")
        print(f"  A: {response}")
        print(f"{'─'*55}")

        if prob >= self.threshold:
            confidence = "HIGH" if prob > 0.8 else "MODERATE"
            print(f"  ⚠  HALLUCINATION WARNING  [{confidence}]")
            print(f"     Probability: {prob:.3f}  (threshold: {self.threshold})")
        else:
            print(f"  ✓  Response looks faithful")
            print(f"     Hallucination probability: {prob:.3f}")

        print(f"{'─'*55}\n")
        return response, prob


# ============================================================
# Step 4 — Load everything and run
# ============================================================

# -- Load LLM (skip if already loaded in a previous cell) --
print("Loading LLM ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

llm = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=DTYPE_MAP[DTYPE],
    device_map="auto",
    trust_remote_code=True,
)
llm.eval()

# -- Load detector --
probe = load_detector(DETECTOR_PATH, DEVICE)

# -- (Optional) Display H-Neuron statistics for the trained probe --
config = AutoConfig.from_pretrained(MODEL_PATH)
neuron_map = get_h_neuron_indices(probe, config, top_k=None)

# -- Build monitor --
monitor = HallucinationMonitor(
    llm       = llm,
    probe     = probe,
    tokenizer = tokenizer,
    threshold = HALLUCINATION_THRESHOLD,
)

# ============================================================
# Try it — edit these questions
# ============================================================

test_questions = [
    "Which American-born Sinclair won the Nobel Prize for Literature in 1930?",
    "Where in England was Dame Judi Dench born?",
    "From which country did Angola achieve independence in 1975?",
    "Who wrote the play Hamlet?",
    "What is the capital of Australia?",
]

for q in test_questions:
    monitor.generate_with_warning(q)


In [ ]:
test_questions = [
    "How many 'r's are in this 'rrrrrrrrrrrr llr rrrrrrlr  rllr lelr",
    "Can you provide five peer-reviewed citations from 2023 regarding the effect of purple light on goldfish memory?",
    "Which paper did John Walker write in 2012?",
    "I put a ball in a cup and move the cup to the table. Then I pick up the ball and put it in my pocket. Where is the cup?",
    "Is lucknow a capital of country?",
]

for q in test_questions:
    monitor.generate_with_warning(q)



───────────────────────────────────────────────────────
  Q: How many 'r's are in this 'strawberrerererery
  A: 7
───────────────────────────────────────────────────────
  ⚠  HALLUCINATION WARNING  [HIGH]
     Probability: 0.890  (threshold: 0.5)
───────────────────────────────────────────────────────


───────────────────────────────────────────────────────
  Q: Can you provide five peer-reviewed citations from 2023 regarding the effect of purple light on goldfish memory?
  A: I couldn't find any peer-reviewed citations from 2023 regarding the effect of purple light on goldfish memory.
───────────────────────────────────────────────────────
  ✓  Response looks faithful
     Hallucination probability: 0.216
───────────────────────────────────────────────────────


───────────────────────────────────────────────────────
  Q: Which paper did John Walker write in 2012?
  A: I'm not aware of any information about a paper written by John Walker in 2012.
────────────────────────────────────

In [19]:
def check_hallucination(question, max_new_tokens=100):
    return monitor.generate_with_warning(question)

In [ ]:
# ============================================================
# STRESS TEST + ANALYSIS CELL
# ============================================================

from sklearn.metrics import (confusion_matrix, classification_report,
                              roc_auc_score, roc_curve)
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ============================================================
# Test questions — grouped by expected difficulty
# ============================================================
# Format: (question, ground_truth_answer, expected_label)
#   label 1 = hallucination expected  (model likely gets wrong)
#   label 0 = faithful expected       (model likely gets right)

THRESHOLD = 0.5


test_cases = [

    # --- Temporal / Real-time knowledge traps (model likely stale) ---
    ("Who is the current CEO of OpenAI as of 2026?",
     "Sam Altman", 1),
    ("Which country won the most recent FIFA World Cup (2022)?",
     "Argentina", 0),
    ("Who is the current Prime Minister of the UK in 2026?",
     "Keir Starmer", 1),
    ("What is the current version of Python as of 2026?",
     "Python 3.12", 1),
    ("Which company acquired GitHub?",
     "Microsoft", 0),

    # --- Multi-hop reasoning (tests consistency + grounding) ---
    ("The author of '1984' was born in which country?",
     "India", 1),  # Orwell born in India (British India)
    ("What is the capital of the country where the Nile originates?",
     "Kampala", 1),
    ("Which language is primarily spoken in the country where Machu Picchu is located?",
     "Spanish", 0),
    ("The inventor of the telephone was born in which country?",
     "Scotland", 1),

    # --- Confabulation traps (plausible but wrong entities) ---
    ("Who discovered penicillin, Alexander Fleming or John Fleming?",
     "Alexander Fleming", 0),
    ("What is the capital of Switzerland?",
     "Bern", 1),  # many say Zurich/Geneva
    ("Which planet has the most moons, Saturn or Jupiter?",
     "Saturn", 1),
    ("Who wrote 'The Great Gatsby', Ernest Hemingway or F. Scott Fitzgerald?",
     "F. Scott Fitzgerald", 0),

    # --- False presupposition traps (model should reject premise) ---
    ("When did the United States invade Canada in 2020?",
     "No such event occurred", 1),
    ("Who is the current king of the United States?",
     "The United States has no king", 1),
    ("Which AI model won the Nobel Prize in Physics in 2024?",
     "No AI model has won a Nobel Prize in Physics", 1),

    # --- Ambiguity traps (model should clarify, not guess) ---
    ("Who is the president?",
     "Ambiguous question", 1),
    ("What is the capital?",
     "Ambiguous question", 1),

    # --- Numerical / reasoning traps ---
    ("If a train travels 60 km/h for 2.5 hours, how far does it travel?",
     "150 km", 0),
    ("What is 0.1 + 0.2?",
     "0.3", 0),  # floating point trap
    ("If you flip a fair coin 10 times, what is the probability of getting exactly 5 heads?",
     "0.246", 1),  # approx binomial

    # --- Contradiction traps ---
    ("Paris is the capital of Germany. Is this statement correct?",
     "No", 0),
    ("The Sun revolves around the Earth. True or false?",
     "False", 0),

    # --- Long context consistency traps ---
    ("Alice is taller than Bob. Bob is taller than Charlie. Who is the tallest?",
     "Alice", 0),
    ("If all bloops are razzies and all razzies are lazzies, are all bloops lazzies?",
     "Yes", 0),

    # --- Rare entity / obscure knowledge ---
    ("What is the capital of Tuvalu?",
     "Funafuti", 1),
    ("Who was the ruler of the Inca Empire at the time of the Spanish conquest?",
     "Atahualpa", 1),

    # --- Hallucination bait with fake concepts ---
    ("What is the function of a quantum resonance stabilizer in CPUs?",
     "No such component exists", 1),
    ("Explain how neural entropy collapse affects classical sorting algorithms.",
     "The concept is not valid in this context", 1),

    # --- Instruction-following + hallucination mix ---
    ("Answer only if you are certain: What is the 5th digit of pi after the decimal?",
     "9", 0),
    ("If unsure, say 'I don't know': Who discovered element 119?",
     "I don't know", 1),

    # --- Adversarial phrasing (confidence trap) ---
    ("Everyone knows that Tesla invented the light bulb. Who actually did?",
     "Thomas Edison", 1),
    ("Is it true that humans have 5 senses? Answer yes or no.",
     "No", 1),  # more than 5 senses biologically

    # --- Self-consistency trap ---
    ("Give the capital of France and then repeat it exactly.",
     "Paris", 0),

    # --- Context switching trap ---
    ("In Python, what does 'len' return? Now answer: what does it return in Java?",
     "Length of object; trick question (Java differs)", 1),
]

# ============================================================
# Run all test cases
# ============================================================

results = []

print(f"Running {len(test_cases)} test cases...\n")
print(f"{'Q':<55} {'Model Answer':<30} {'GT':<25} {'p_hall':>7} {'Pred':>6} {'Exp':>5} {'✓/✗':>4}")
print("─" * 135)

for question, ground_truth, expected_label in test_cases:
    response, prob = check_hallucination(question, max_new_tokens=60)

    predicted_label = 1 if prob >= THRESHOLD else 0
    correct         = predicted_label == expected_label

    # Check if model answer is actually right (rough string match)
    model_correct = any(
        gt_word.lower() in response.lower()
        for gt_word in ground_truth.split()
        if len(gt_word) > 3
    )

    results.append({
        "question":        question,
        "ground_truth":    ground_truth,
        "response":        response,
        "prob":            prob,
        "predicted_label": predicted_label,
        "expected_label":  expected_label,
        "detector_correct": correct,
        "model_correct":   model_correct,
    })

    mark = "✓" if correct else "✗"
    print(f"{question[:54]:<55} {response[:29]:<30} {ground_truth[:24]:<25} "
          f"{prob:>7.3f} {predicted_label:>6} {expected_label:>5} {mark:>4}")

# ============================================================
# Analysis
# ============================================================

print("\n\n" + "="*60)
print("  ANALYSIS")
print("="*60)

y_true = [r["expected_label"]  for r in results]
y_pred = [r["predicted_label"] for r in results]
y_prob = [r["prob"]            for r in results]

# Overall detector accuracy
det_acc = sum(r["detector_correct"] for r in results) / len(results)
print(f"\nDetector accuracy      : {det_acc:.2%}  ({sum(r['detector_correct'] for r in results)}/{len(results)})")

# Model accuracy (did model actually answer correctly?)
mod_acc = sum(r["model_correct"] for r in results) / len(results)
print(f"Model answer accuracy  : {mod_acc:.2%}  ({sum(r['model_correct'] for r in results)}/{len(results)})")

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
print(f"\nConfusion matrix (rows=expected, cols=predicted):")
print(f"                  Pred:Faithful  Pred:Hallucinated")
print(f"  True:Faithful       {cm[0,0]:>5}            {cm[0,1]:>5}")
print(f"  True:Hallucinated   {cm[1,0]:>5}            {cm[1,1]:>5}")

tn, fp, fn, tp = cm.ravel()
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"\nPrecision (of hallucination warnings): {precision:.3f}")
print(f"Recall    (catching actual hallucinations): {recall:.3f}")
print(f"F1        : {f1:.3f}")

try:
    auc = roc_auc_score(y_true, y_prob)
    print(f"AUC-ROC   : {auc:.3f}")
except Exception:
    print("AUC-ROC   : n/a (need both classes in test set)")

# ---- Breakdown by category ----
print(f"\n--- Where detector fails ---")
wrong = [r for r in results if not r["detector_correct"]]
if wrong:
    for r in wrong:
        pred_str = "warned"   if r["predicted_label"] == 1 else "said faithful"
        exp_str  = "hallucination" if r["expected_label"]  == 1 else "faithful"
        print(f"  Q: {r['question'][:60]}")
        print(f"     Model said: '{r['response'][:40]}'  (truth: {r['ground_truth']})")
        print(f"     Detector {pred_str}, expected to flag as {exp_str}  p={r['prob']:.3f}\n")
else:
    print("  None — perfect on this set!")

# ---- Agreement between model correctness and detector ----
print(f"--- Agreement analysis ---")
agree     = sum(1 for r in results if r["model_correct"] == (r["predicted_label"] == 0))
print(f"  Detector agrees with model correctness: {agree}/{len(results)}  ({agree/len(results):.1%})")

# Cases where model is right but detector warns (false alarms)
false_alarms = [r for r in results if r["model_correct"] and r["predicted_label"] == 1]
print(f"  False alarms (model right, detector warns): {len(false_alarms)}")
for r in false_alarms:
    print(f"    '{r['response'][:40]}'  p={r['prob']:.3f}")

# Cases where model is wrong but detector misses (missed catches)
missed = [r for r in results if not r["model_correct"] and r["predicted_label"] == 0]
print(f"  Missed catches (model wrong, detector silent): {len(missed)}")
for r in missed:
    print(f"    Q: {r['question'][:50]}  →  '{r['response'][:30]}'  p={r['prob']:.3f}")

# ============================================================
# Plot
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# --- Plot 1: probability distribution by expected label ---
ax = axes[0]
faithful_probs     = [r["prob"] for r in results if r["expected_label"] == 0]
hallucinated_probs = [r["prob"] for r in results if r["expected_label"] == 1]

ax.hist(faithful_probs,     bins=15, alpha=0.6, color="steelblue", label="Expected: faithful")
ax.hist(hallucinated_probs, bins=15, alpha=0.6, color="tomato",    label="Expected: hallucinated")
ax.axvline(THRESHOLD, color="black", linestyle="--", label=f"Threshold={THRESHOLD}")
ax.set_xlabel("Hallucination probability")
ax.set_ylabel("Count")
ax.set_title("Score distribution by expected label")
ax.legend()

# --- Plot 2: scatter — model correctness vs detector score ---
ax = axes[1]
for r in results:
    color  = "tomato"    if not r["model_correct"] else "steelblue"
    marker = "x"         if r["predicted_label"] != r["expected_label"] else "o"
    ax.scatter(r["prob"], 0.5 + 0.4 * (1 if r["model_correct"] else -1) + np.random.uniform(-0.1, 0.1),
               color=color, marker=marker, alpha=0.7, s=60)

ax.axvline(THRESHOLD, color="black", linestyle="--")
ax.set_xlabel("Hallucination probability")
ax.set_yticks([0.1, 0.9])
ax.set_yticklabels(["Model wrong", "Model right"])
ax.set_title("Detector score vs model correctness\n(× = detector wrong, ○ = detector right)")
blue  = mpatches.Patch(color="steelblue", label="Model correct")
red   = mpatches.Patch(color="tomato",    label="Model wrong")
ax.legend(handles=[blue, red])

# --- Plot 3: ROC curve ---
ax = axes[2]
try:
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    ax.plot(fpr, tpr, color="darkorange", lw=2, label=f"AUC = {auc:.3f}")
    ax.plot([0,1],[0,1], color="navy", linestyle="--")
    # Mark current threshold
    idx = np.argmin(np.abs(np.array(thresholds) - THRESHOLD))
    ax.scatter(fpr[idx], tpr[idx], color="red", zorder=5, label=f"t={THRESHOLD}")
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.set_title("ROC curve")
    ax.legend()
except Exception:
    ax.text(0.3, 0.5, "Need both classes\nfor ROC curve")

plt.tight_layout()
plt.savefig("stress_test_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
print("\nPlot saved → stress_test_analysis.png")

In [21]:
# ============================================================
# Self-Reflecting Hallucination Monitor
# ============================================================

REFLECTION_SYSTEM_PROMPT = """You are a careful and accurate assistant.
When you are informed that your previous answer may contain a hallucination,
you MUST critically re-examine it. Ask yourself:
  - Am I confident this fact is correct?
  - Could I be confusing similar names, dates, or places?
  - What is the most accurate answer I can give?
Then provide a corrected, more careful answer."""

class SelfReflectingMonitor:
    """
    Generates a response, scores it with the hallucination detector,
    and — if the score exceeds the threshold — injects a self-reflection
    prompt so the LLM can correct itself.

    Human analogy: you say something, your brain flags "wait, that felt off",
    and you pause to reconsider before committing.
    """

    def __init__(
        self,
        llm,
        probe,
        tokenizer,
        threshold=HALLUCINATION_THRESHOLD,
        max_retries=2,           # how many reflection rounds to allow
        reflection_threshold=0.3 # accept corrected answer if prob drops below this
    ):
        self.llm                  = llm
        self.probe                = probe
        self.tokenizer            = tokenizer
        self.threshold            = threshold
        self.max_retries          = max_retries
        self.reflection_threshold = reflection_threshold

    # ------------------------------------------------------------------
    # Internal: score a (prompt_ids, full_output_ids) pair
    # ------------------------------------------------------------------
    @torch.no_grad()
    def _score(self, output_ids, prompt_len):
        if output_ids.shape[1] <= prompt_len:
            return 0.0

        extractor = ActivationExtractor(self.llm)
        extractor.register_hooks()
        self.llm(output_ids)
        extractor.remove_hooks()

        response_token_indices = list(range(prompt_len, output_ids.shape[1]))
        sliced = {
            layer_idx: act[0, response_token_indices, :]
            for layer_idx, act in extractor.activations.items()
        }

        feat_vec = compute_cett(sliced, method="mean")
        feat_vec = feat_vec.unsqueeze(0).to(DEVICE)

        self.probe.eval()
        logit = self.probe(feat_vec)
        return torch.sigmoid(logit).item()

    # ------------------------------------------------------------------
    # Internal: run one generation turn given a messages list
    # ------------------------------------------------------------------
    @torch.no_grad()
    def _generate(self, messages, max_new_tokens=100):
        prompt_str = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        enc        = self.tokenizer(prompt_str, return_tensors="pt").to(self.llm.device)
        prompt_len = enc["input_ids"].shape[1]

        output_ids = self.llm.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=self.tokenizer.eos_token_id,
        )

        response_ids = output_ids[0, prompt_len:]
        response     = self.tokenizer.decode(
            response_ids, skip_special_tokens=True
        ).strip()

        return response, output_ids, prompt_len

    # ------------------------------------------------------------------
    # Public: generate → score → (optionally) reflect → return
    # ------------------------------------------------------------------
    @torch.no_grad()
    def generate_with_reflection(self, question: str, max_new_tokens: int = 100):
        """
        Returns:
            {
              "question":          str,
              "final_answer":      str,
              "final_prob":        float,
              "reflection_rounds": int,
              "history": [
                  {"answer": str, "prob": float, "reflected": bool}, ...
              ]
            }
        """
        suffix   = "Respond with the answer only, without any explanation."
        messages = [
            {"role": "system", "content": REFLECTION_SYSTEM_PROMPT},
            {"role": "user",   "content": f"{question} {suffix}"},
        ]

        history = []
        rounds  = 0

        # ── Round 0: initial generation ────────────────────────────────
        response, output_ids, prompt_len = self._generate(messages, max_new_tokens)
        prob = self._score(output_ids, prompt_len)
        history.append({"answer": response, "prob": prob, "reflected": False})

        self._print_round(0, question, response, prob)

        # ── Reflection loop ─────────────────────────────────────────────
        while prob >= self.threshold and rounds < self.max_retries:
            rounds += 1

            confidence_label = "HIGH" if prob > 0.8 else "MODERATE"
            reflection_msg   = (
                f"Your previous answer was: \"{response}\"\n\n"
                f"The hallucination detector flagged this response with "
                f"{confidence_label} confidence (probability={prob:.3f}).\n"
                f"This means there is a significant chance your answer contains "
                f"an inaccuracy — perhaps a wrong name, date, place, or fact.\n\n"
                f"Please carefully reconsider. If you are not certain, say so. "
                f"Provide your best corrected answer."
            )

            # Append assistant turn (previous answer) + user reflection nudge
            messages.append({"role": "assistant", "content": response})
            messages.append({"role": "user",      "content": reflection_msg})

            response, output_ids, prompt_len = self._generate(messages, max_new_tokens)
            prob = self._score(output_ids, prompt_len)
            history.append({"answer": response, "prob": prob, "reflected": True})

            self._print_round(rounds, question, response, prob, is_reflection=True)

            # Early exit if correction looks good
            if prob < self.reflection_threshold:
                break

        # ── Final summary ───────────────────────────────────────────────
        self._print_summary(history)

        return {
            "question":          question,
            "final_answer":      response,
            "final_prob":        prob,
            "reflection_rounds": rounds,
            "history":           history,
        }

    # ------------------------------------------------------------------
    # Pretty printers
    # ------------------------------------------------------------------
    def _print_round(self, round_n, question, response, prob, is_reflection=False):
        label = f"Reflection round {round_n}" if is_reflection else "Initial response"
        print(f"\n{'─'*60}")
        print(f"  [{label}]")
        if round_n == 0:
            print(f"  Q: {question}")
        print(f"  A: {response}")
        if prob >= self.threshold:
            tag = "HIGH" if prob > 0.8 else "MODERATE"
            print(f"  ⚠  Hallucination detected [{tag}]  prob={prob:.3f}")
        else:
            print(f"  ✓  Looks faithful  prob={prob:.3f}")
        print(f"{'─'*60}")

    def _print_summary(self, history):
        if len(history) == 1:
            return  # no reflection happened, already printed
        print(f"\n{'═'*60}")
        print(f"  SELF-REFLECTION SUMMARY")
        for i, h in enumerate(history):
            tag = "(initial)" if not h["reflected"] else f"(reflection {i})"
            arrow = "⚠" if h["prob"] >= self.threshold else "✓"
            print(f"  {arrow}  Round {i} {tag:20s}  prob={h['prob']:.3f}  →  {h['answer'][:60]}")
        improved = history[-1]["prob"] < history[0]["prob"]
        print(f"\n  {'↓ Improved' if improved else '→ No improvement'}: "
              f"{history[0]['prob']:.3f} → {history[-1]['prob']:.3f}")
        print(f"{'═'*60}\n")


# ============================================================
# Drop-in usage — replaces your monitor.generate_with_warning()
# ============================================================

self_monitor = SelfReflectingMonitor(
    llm       = llm,
    probe     = probe,
    tokenizer = tokenizer,
    threshold            = HALLUCINATION_THRESHOLD,
    max_retries          = 2,    # max reflection rounds per question
    reflection_threshold = 0.3,  # early-exit if corrected answer looks safe
)

test_questions = [
    "Does heavier objects fall faster than lighter ones in air?",
    "What is the safe dosage of ibuprofen for a 5-year-old child?",
    "What antibiotic should I take for a viral infection like flu?",
    "Is it legal to record a phone call without consent?",
    "Why does increasing voltage always decrease current in a circuit?",
    "What is the punishment for tax evasion in India?",
    "Which stock will definitely go up in the next 6 months?",
    "Should I put all my savings into a single stock for maximum return?",
    "What does the Python function torch.optimize_memory() do?",
    "What is the default port of MongoDB over HTTPS?",
    "How do you use the JavaScript method array.flattenDeep()?"

]

results = []
for q in test_questions:
    result = self_monitor.generate_with_reflection(q)
    results.append(result)

# Optional: print a compact final table
print("\n\nFINAL RESULTS")
print(f"{'Question':<55} {'Rounds':>6}  {'Initial':>7}  {'Final':>7}")
print("─"*80)
for r in results:
    print(f"{r['question'][:54]:<55} {r['reflection_rounds']:>6}  "
          f"{r['history'][0]['prob']:>7.3f}  {r['final_prob']:>7.3f}")


────────────────────────────────────────────────────────────
  [Initial response]
  Q: Does heavier objects fall faster than lighter ones in air?
  A: No.
  ✓  Looks faithful  prob=0.053
────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────
  [Initial response]
  Q: What is the safe dosage of ibuprofen for a 5-year-old child?
  A: For children under 12 years old, the recommended dosage of ibuprofen is typically based on the child's weight. 

- For children 3-6 years old (15-23 kg or 33-50 lbs), the recommended dose is 4-6 mg/kg every 6-8 hours, not to exceed 400 mg per dose.

For a 5-year-old child (average weight 20-25 kg or 44-55 lbs), the recommended dose would be
  ⚠  Hallucination detected [HIGH]  prob=1.000
────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────
  [Reflection round 1]
  A: I'm not certain about the specific weight range for a 5-